In [1]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import json


In [2]:
load_dotenv(override=True)
ollama_url = os.getenv("OLLAMA_BASE_URL")

In [ ]:
links_system_prompt = """IMPORTANT:
    - OUTPUT ONLY JSON.
    - NO explanations.
    - NO summaries.
    - FOLLOW THIS STRUCTURE EXACTLY:

    {
      "relevant_links": [
        {"type": "about page", "url": "https://example.com/about"}
      ]
    }
"""
links_system_prompt

'Return relevant links for a sales brochure from given list of links. Do not summarize or explain the links. Only return links that would be useful for creating a sales brochure, such as \'about us\', \'products\', \'services\', \'testimonials\', and \'contact\' pages. Ignore links related to blogs, news, privacy policies, terms of service, or unrelated content.\nOutput a JSON response in the following format:\n\n{\n  "relevant_links": [\n    {"type": "about page", "url": "https://example.com/about"},\n    {"type": "careers page", "url": "https://example.com/careers"}\n  ]\n}\n'

In [ ]:
def select_relevant_links(url):
    client = OpenAI(base_url=ollama_url, api_key="")
    response = client.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": links_system_prompt},
            {"role": "user", "content": "Return json for links: " + " ".join(fetch_website_links(url))}
        ]
    )
    return response.choices[0].message.content

In [15]:
select_relevant_links("https://timesofindia.indiatimes.com")

'Okay, here\'s a breakdown of the provided Times of India content, categorized for clarity:\n\n**1. Breaking News & Current Events:**\n\n*   **India Focused:**\n    *   **BSP and Odisha Police Recover Arms Cache:** Significant seizure of Maoist weapons in Malkangiri district, Odisha, involving BSF and Odisha Police. (Multiple Articles)\n    *   **US-India Trade Deal Closer:**  Negotiations regarding a US-India trade deal are nearing completion, expected to be highly detailed\n    *   **Cyber Breach concerns:** PwC flags 1 million Indian firms are potentially exposed to cyber breaches\n*   **International:**\n    *   **Sheikh Hasina Sentenced to Death:** Bangladeshi Prime Minister Sheikh Hasina has been sentenced to death for crimes against humanity. (Significant global reaction expected)\n    *   **Ukraine Bombing:** Russia continues to target Ukraine\'s second-largest city, Kharkiv.\n    *   **US-India Trade Advance:** Approaching agreement on trade between the US and India.\n\n**2. E

In [6]:
brochure_system_prompt = """You are provided with the main content and relevant links of a website.
Create a compelling sales brochure for the company represented by the website.
Write in a way that its geared towards 10 year olds, highlighting the company's strengths, products, and services.
"""

In [ ]:
brochure_user_prompt = """Website Content:
{website_content}
Relevant Links:
{relevant_links}
"""

In [ ]:
client = OpenAI(base_url=ollama_url, api_key="")

def create_sales_brochure(website_content, relevant_links):
    stream = client.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": brochure_user_prompt.format(website_content=website_content, relevant_links=relevant_links)}
        ],
        stream=True
    )
    return stream

In [ ]:
responseStream = create_sales_brochure(fetch_website_contents("https://timesofindia.indiatimes.com"), select_relevant_links("https://timesofindia.indiatimes.com"))
display(Markdown("### Sales Brochure:\n"))
responseText = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in responseStream:
    responseText += chunk.choices[0].delta.get("content", "")
    display_handle.update(Markdown(responseText), display_id = display_handle.display_id)
    

In [13]:
fetch_website_links("https://timesofindia.indiatimes.com")

['https://timesofindia.indiatimes.com/',
 'https://timesofindia.indiatimes.com/us',
 'https://timesofindia.indiatimes.com/',
 'https://navbharattimes.indiatimes.com/',
 'https://marathi.indiatimes.com/',
 'https://vijaykarnataka.com/',
 'https://tamil.samayam.com/',
 'https://bangla.indiatimes.com/',
 'https://malayalam.samayam.com/',
 'https://telugu.samayam.com/',
 'https://www.iamgujarat.com/',
 'https://timesofindia.indiatimes.com/weather',
 'https://timesofindia.indiatimes.com',
 'https://timesofindia.indiatimes.com/toi-plus',
 'https://timesofindia.indiatimes.com/games?utm_source=top_nav&utm_campaign=game_promotion',
 'https://timesofindia.indiatimes.com/videos',
 'https://timesofindia.indiatimes.com/city',
 'https://timesofindia.indiatimes.com/city/mumbai',
 'https://timesofindia.indiatimes.com/city/delhi',
 'https://timesofindia.indiatimes.com/city/bangalore',
 'https://timesofindia.indiatimes.com/city/hyderabad',
 'https://timesofindia.indiatimes.com/city/kolkata',
 'https://t